# Model 10: evidence-calibrated uncertainty envelopes

This notebook separates published numerical constraints from modeling conventions and parameters that remain unconstrained. Its main result is a **required post-contact mixing threshold**, not a posterior probability that the founder scenario occurred.

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = None
if IN_COLAB:
    REPO_DIR = Path('/content/Evolution-Creation')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git',str(REPO_DIR)],check=True)
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','-q','origin','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'checkout','-q','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev]'],check=True)
else:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'src' / 'evolution_creation').exists():
            REPO_DIR = candidate
            break

if REPO_DIR is not None:
    src_path = str(REPO_DIR / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print('Environment ready:', REPO_DIR if REPO_DIR is not None else 'using installed Python environment')


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.evidence_calibration import (
    BYARD_POPULATION_MODEL_SET, contact_generations, preclosure_generations,
    required_external_parent_rate, sample_evidence_envelope,
    simulate_bottleneck_fixation_probability, simulate_fixation_rates,
)


## Evidence ledger

The repository stores source status in `data/model10_evidence.json`. The external-parent probability is intentionally marked **unconstrained** because the reviewed evidence does not supply a Tasmania-specific per-generation estimate.

In [ ]:
ledger_path = (REPO_DIR / 'data' / 'model10_evidence.json') if REPO_DIR is not None else Path('data/model10_evidence.json')
if not ledger_path.exists(): ledger_path = Path('../data/model10_evidence.json')
print(json.dumps(json.loads(ledger_path.read_text()),indent=2)[:12000] if ledger_path.exists() else 'Evidence ledger not found')


## Chronology

Published generation-time evidence supports roughly 26-30 years per generation. The direct Bassian land-bridge bracket used here is 11,960-12,890 BP. With an 11 ka founder date outside Tasmania, the direct bracket gives zero pre-closure generations.

In [ ]:
generation_interval=widgets.FloatSlider(value=28,min=26,max=30,step=.1,description='Years/gen')
isolation_age=widgets.IntSlider(value=12000,min=9000,max=13500,step=50,description='Isolation BP')
contact_year=widgets.Dropdown(options=[1797,1803],value=1797,description='Contact year')
display(generation_interval,isolation_age,contact_year)
def chronology(_=None):
    gi=generation_interval.value
    print('11 ka founder depth:',round(11000/gi),'generations')
    print('Pre-closure generations:',preclosure_generations(11000,isolation_age.value,gi))
    print('Contact generations to 2026:',contact_generations(contact_year.value,2026,gi))
widgets.interactive_output(chronology,{'_':widgets.fixed(None)}); chronology()


## Required mixing threshold

The threshold assumes the external pool is already 100% descended from both founders. It is deliberately favorable to the founder scenario.

In [ ]:
population=widgets.Dropdown(options=[int(x) for x in BYARD_POPULATION_MODEL_SET],value=7465,description='Population')
generations=widgets.IntSlider(value=8,min=5,max=12,description='Generations')
target=widgets.FloatSlider(value=.95,min=.1,max=.99,step=.01,description='Fixation P')
display(population,generations,target)
def threshold(_=None):
    r=required_external_parent_rate(target.value,generations.value,population.value)
    print(f'Required external-parent probability = {100*r:.3f}% per parental draw')
widgets.interactive_output(threshold,{'_':widgets.fixed(None)}); threshold()


In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
for g,ls in [(8,'-'),(9,'--')]:
    for p in [.5,.95]:
        y=[100*required_external_parent_rate(p,g,int(n)) for n in BYARD_POPULATION_MODEL_SET]
        ax.plot(BYARD_POPULATION_MODEL_SET,y,linestyle=ls,marker='o',label=f'{int(100*p)}% fixation, {g} generations')
ax.set(xlabel='Published contact-era population model output',ylabel='Required external-parent probability (%)'); ax.legend(); plt.show()


## Evidence-envelope propagation

Uniform draws inside published intervals and equal weighting of the nine population outputs are propagation conventions, not paper-supplied posterior distributions.

In [ ]:
env=sample_evidence_envelope(samples=20000,contact_start_year=contact_year.value,seed=20260920)
fig,ax=plt.subplots(figsize=(10,5))
ax.hist(100*env.required_rate_50,bins=40,alpha=.6,label='50% fixation threshold')
ax.hist(100*env.required_rate_95,bins=40,alpha=.6,label='95% fixation threshold')
ax.set(xlabel='Required external-parent probability (%)',ylabel='Envelope samples'); ax.legend(); plt.show()
print('Founder generations 5/50/95%:',np.quantile(env.founder_generations,[.05,.5,.95]))
print('Contact generation counts:',np.unique(env.late_contact_generations,return_counts=True))
print('Maximum pre-closure generations:',env.preclosure_generations.max())


## Monte Carlo validation and bottleneck sensitivity

Shared pedigree makes the analytic independence threshold approximate. The first plot checks it directly. The second is only a demographic sensitivity experiment.

In [ ]:
n=population.value; g=generations.value
r50=required_external_parent_rate(.5,g,n); r95=required_external_parent_rate(.95,g,n)
rates=np.unique(np.sort([.85*r50,r50,1.1*r50,.9*r95,r95,1.1*r95]))
mc=simulate_fixation_rates(rates,g,n,replicates=5000,seed=20260920)
fig,ax=plt.subplots(figsize=(9,5)); ax.plot(100*rates,mc.fixation_probability,marker='o'); ax.axhline(.5,ls='--'); ax.axhline(.95,ls='--'); ax.set(xlabel='External-parent probability (%)',ylabel='Monte Carlo fixation probability',ylim=(0,1.02)); plt.show()
rates=np.linspace(.005,.035,13); constant=[]; bottleneck=[]
for i,r in enumerate(rates):
    constant.append(simulate_bottleneck_fixation_probability(r,[7465]*9,replicates=3000,seed=100+i))
    bottleneck.append(simulate_bottleneck_fixation_probability(r,[7465,342,342,342,342,342,342,342,342],replicates=3000,seed=200+i))
fig,ax=plt.subplots(figsize=(9,5)); ax.plot(100*rates,constant,marker='o',label='constant N=7,465'); ax.plot(100*rates,bottleneck,marker='o',label='after first generation N=342'); ax.set(xlabel='External-parent probability (%)',ylabel='Fixation probability',ylim=(0,1.02)); ax.legend(); plt.show()


## Interpretation

The evidence constrains chronology and plausible generation count much better than it constrains reproductive mixing. Model 10 therefore reports threshold curves, not a historical probability of universal ancestry.